In [3]:

import os 
import pandas as pd
import json
from sqlalchemy import create_engine
import sqlite3

In [2]:
engine = create_engine("sqlite:///Restautrant.db")

In [8]:
folder_path = r"C:\Users\satvi\Downloads\Yelp-JSON\Yelp JSON\yelp_dataset"

for file in os.listdir(folder_path):
    if file.endswith('.json'):
        table_name = file.replace('.json','')
        print(f"Loading {file}.....")
        
        first_chunk = True
        
        for chunk in pd.read_json(
            os.path.join(folder_path,file),
            lines=True,
            chunksize = 100000
        ):
            for col in chunk.columns:
                if chunk[col].apply(lambda x: isinstance(x,dict)).any():
                    chunk[col] = chunk[col].apply(
                        lambda x: json.dumps(x) if isinstance(x,dict) else x
                    )
                
            chunk.to_sql(
                table_name,
                engine,
                if_exists = 'replace' if first_chunk else 'append',
                index = False
            )
            
            first_chunk = False
        
        print(f"loaded {table_name}")

Loading yelp_academic_dataset_business.json.....
loaded yelp_academic_dataset_business
Loading yelp_academic_dataset_checkin.json.....
loaded yelp_academic_dataset_checkin
Loading yelp_academic_dataset_review.json.....
loaded yelp_academic_dataset_review
Loading yelp_academic_dataset_tip.json.....
loaded yelp_academic_dataset_tip
Loading yelp_academic_dataset_user.json.....
loaded yelp_academic_dataset_user


In [4]:
conn = sqlite3.connect('Restautrant.db')

In [5]:
pd.read_sql(
    "SELECT * FROM yelp_academic_dataset_user LIMIT 5",
    conn
)

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,...,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,...,1,0,0,0,1,1,0,0,0,0
